In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

import sklearn
sklearn.set_config(display='text')
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

from sklearn.neighbors import KNeighborsClassifier

머신러닝의 학습 방법은 지도 학습(supervised learning)과 비지도 학습(unsupervised learning)으로 나눌 수 있다.

지도 학습이란 라벨링이된 데이터를 학습시키는 것을 의미하며, 비지도 학습은 라벨링이 되지 않은 데이터를 학습시키는 것이다. 이때, 라벨링이란 학습 데이터에 정답이 표시된 것을 의미하며, 정답 부분을 레이블(실제값, 타겟, 클래스) 데이터라고 한다.

레이블의 형태에 따라 지도 학습은 세부적으로 두 가지 종류로 나눌 수 있다. 레이블이 범주형인 경우에는 분류(classification) 문제라고 하고, 연속형 숫자인 경우에는 회귀(regression) 문제라고 한다. 머신러닝 알고리즘을 적용하기 전에 레이블의 형태를 파악하고 풀려는 문제가 분류 문제인지 회귀 문제인지 파악하는 것이 중요하다.

`k-최근접 이웃(k-Nearest Neighbor, kNN) 알고리즘`  

k-최근접 이웃 알고리즘은 이해하기 쉽고, 자주 사용되는 방법으로 비교 대상이 되는 데이터 포인트 주변에 가장 가까이 존재하는 k개의 데이터와 비교해서 가장 가까운 데이터 종류로 판별한다.

k-최근접 이웃 알고리즘은 학습 과정에서 게으른 학습(lazy learning) 방법을 사용한다.  
게으른 학습은 학습 데이터 전체를 메모리상에 보관하면서 테스트 데이터가 새로 들어왔을 때 바로 학습하는 것을 의미한다. 학습 데이터를 메모리에 보관하므로 추가적인 학습 시간없이 곧바로 학습 결과를 얻을 수 있다는 장점이 있지만 데이터가 지나치게 커서 메모리에 보관할 수 없을 경우 사용할 수 없다는 단점이 있다.

게으른 학습의 반대말은 열정적 학습(eager learning)이라 한다.  
학습 데이터로 일정 기간 학습시킨 후 학습시킨 모델을 기반으로 테스트에 적용하는 방법으로 학습 데이터는 학습시에만 메모리에 보관되며 학습 이후에는 테스트 데이터를 분류, 예측할 때 메모리에 보관할 필요가 없다. 게으른 학습과 열정적 학습의 차이는 학습 시간의 필요 유무에 따라 나뉘고 열정적 학습은 게으른 학습보다 메모리를 효율적으로 사용할 수 있다는 장점이 있지만 학습 시간이 오래 걸린다는 단점이 있다.

붓꽃 데이터를 사용해 붓꽃 종류를 분류하는 모델을 생성하고 학습시킨다.

In [2]:
# 데이터 불러오기
raw_data = datasets.load_iris() # 사이킷런 라이브러리가 제공하는 붓꽃 데이터를 불러온다.
# print(raw_data)

# 피쳐, 레이블 데이터 저장
xData = raw_data.data # 피쳐 데이터를 저장한다.
yData = raw_data.target # 피쳐 데이터에 따른 레이블을 저장한다.
print(xData.shape, yData.shape)

# 학습 데이터와 테스트 데이터로 분할
# train_test_split() 메소드로 피쳐 데이터와 레이블 데이터를 넘겨서 학습 데이터와 테스트 데이터로 나눈다.
# train_size 속성으로 학습 데이터로 사용할 데이터 비율을 지정한다.
# test_size 속성으로 테스트 데이터로 사용할 데이터 비율을 지정한다.
# train_size, test_size 속성을 생락하면 학습 데이터와 테스트 데이터를 75:25 비율로 분할한다.
# random_state 속성을 지정하면 실행할 때 마다 매번 같은 데이터를 얻을 수 있다. => 항상 같은 결과가 나온다.
x_train, x_test, y_train, y_test = train_test_split(xData, yData, random_state=0)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

# 데이터 표준화(정규화)
scaler = StandardScaler() # 표준화 스케일러 객체를 만든다.
# 스케일러는 학습 데이터를 기반으로 실행해야 하기때문에 학습 데이터는 표준화 후 적용하고 테스트 데이터는 학습 데이터에 표준화된 스케일러에 적용만 시킨다.
# scaler.fit(x_train) # 학습 데이터를 표준화 스케일러로 표준화 한다.
# x_train = scaler.transform(x_train) # 표준화된 스케일러에 학습 데이터를 적용한다.
x_train = scaler.fit_transform(x_train) # 학습 데이터를 표준화 스케일러로 표준화하고 적용한다.
x_test = scaler.transform(x_test) # 테스트 데이터를 학습 데이터로 표준화한 스케일러에 적용한다.

# 모델 생성 후 데이터 학습
# model = KNeighborsClassifier(n_neighbors=5) # n_neighbors 속성에 이웃 개수를 지정해서 k-최근접 이웃 모델을 만든다.
# model.fit(x_train, y_train) # 표준화된 학습 데이터(x_train)와 학습 데이터에 따른 레이블(y_train)을 넘겨서 k-최근접 이웃 모델을 학습시킨다.
model = KNeighborsClassifier(n_neighbors=5).fit(x_train, y_train) # k-최근접 이웃 모델을 만들고 학습시킨다.

(150, 4) (150,)
(112, 4) (38, 4) (112,) (38,)


학습된 모델로 테스트 데이터를 예측한다.

In [3]:
predict = model.predict(x_test) # predict() 함수의 인수로 표준화된 테스트 데이터(x_test)를 넘겨서 k-최근접 이웃 모델을 예측한다.
print(predict)

[2 1 0 2 0 2 0 1 1 1 2 1 1 1 1 0 1 1 0 0 2 1 0 0 2 0 0 1 1 0 2 1 0 2 2 1 0
 2]


학습된 모델을 평가한다.

In [4]:
# 정확도 평가
# accuracy_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 정확도를 계산한다.
accuracy = accuracy_score(y_test, predict)
print(accuracy)

0.9736842105263158


In [5]:
# 정밀도 평가
# precision_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 정밀도를 계산한다.
# average 속성의 기본값은 'binary'이고 'binary'는 레이블의 클래스가 딱 2개(0과 1)일 경우를 의미한다.
# 클래스가 3개 이상이라면 multiclass로 취급되고 None, 'macro', 'weighted' 중의 1개를 선택해서 속성값으로 지정해야 한다.
# None: 각 클래스별 정밀도를 계산한다.
# macro: 정밀도의 산술 평균을 계산한다.
# weighted: 정밀도의 가중 평균을 계산한다.
precision = precision_score(y_test, predict, average=None)
print(precision)

[1.  1.  0.9]


In [6]:
# 재현율 평가
# recall_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 재현율을 계산한다.
# average 속성의 사용 방법은 precision_score() 함수와 같다.
recall = recall_score(y_test, predict, average=None)
print(recall)

[1.     0.9375 1.    ]


In [7]:
# f1 score 평가
# f1_score() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 f1 score를 계산한다.
# average 속성의 사용 방법은 precision_score() 함수와 같다.
f1 = f1_score(y_test, predict, average=None)
print(f1)

[1.         0.96774194 0.94736842]


In [8]:
# 혼동 행렬
# confusion_matrix() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 혼동 행렬을 출력한다.
confusion = confusion_matrix(y_test, predict)
print(confusion)

[[13  0  0]
 [ 0 15  1]
 [ 0  0  9]]


In [9]:
# 분류 리포트
# classification_report() 함수의 인수로 테스트 데이터의 레이블(y_test)과 예측값(predict)을 순서대로 넘겨서 분류 리포트를 출력한다.
classification = classification_report(y_test, predict)
print(classification)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        13
           1       1.00      0.94      0.97        16
           2       0.90      1.00      0.95         9

    accuracy                           0.97        38
   macro avg       0.97      0.98      0.97        38
weighted avg       0.98      0.97      0.97        38

